# TF-IDF Text Baseline From Precomputed CSV (IEMOCAP)

Uses `extracted_features/text/tfidf_features.csv` generated by the extractor notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Could not locate repository root (missing pyproject.toml).')

REPO_ROOT = find_repo_root(Path.cwd())
TFIDF_CSV = REPO_ROOT / 'extracted_features' / 'text' / 'tfidf_features.csv'

if not TFIDF_CSV.exists():
    raise FileNotFoundError(
        f'Missing TF-IDF feature CSV: {TFIDF_CSV}\n'
        'Generate it first from feature_extraction/feature_families/text_tfidf.ipynb.'
    )

df = pd.read_csv(TFIDF_CSV)
feature_cols = [c for c in df.columns if c.startswith('tfidf_')]
if not feature_cols:
    raise ValueError(f'No TF-IDF feature columns found in {TFIDF_CSV}')

train_mask = df['split'] == 'train'
test_mask = df['split'] == 'test'

X_train = df.loc[train_mask, feature_cols]
y_train = df.loc[train_mask, 'emotion']
X_test = df.loc[test_mask, feature_cols]
y_test = df.loc[test_mask, 'emotion']

X_train.shape, X_test.shape


In [ ]:
clf = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    solver='liblinear',
    random_state=42,
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print(f'Accuracy: {acc:.4f}')
print(f'Macro F1:  {f1:.4f}')
print('\nClassification report:\n')
print(classification_report(y_test, y_pred))


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(clf, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)
cv_f1 = cross_val_score(clf, X_train, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)

print(f'CV Accuracy (mean+-std): {np.mean(cv_acc):.4f} +- {np.std(cv_acc):.4f}')
print(f'CV Macro F1 (mean+-std):  {np.mean(cv_f1):.4f} +- {np.std(cv_f1):.4f}')
